In [ ]:
!pip install --quiet minatar imageio-ffmpeg

from google.colab import drive
drive.mount('/content/drive')

import os, numpy as np, torch, torch.nn as nn, torch.optim as optim, random, pickle
from collections import deque, namedtuple
from PIL import Image
import imageio
import matplotlib.pyplot as plt

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/rl-final-project"
print("Project:", DRIVE_PROJECT_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/rl-final-project
Device: cuda


In [ ]:
import minatar

In [ ]:
def preprocess_state(st):
    arr = np.array(st, dtype=np.float32)
    if arr.ndim == 1:
        if arr.size == 1:
            arr = np.full((10,10), arr[0], dtype=np.float32)
        else:
            try:
                arr = arr.reshape(10,10)
            except:
                arr = np.zeros((10,10), dtype=np.float32)
    if arr.ndim == 3:
        arr = np.sum(arr, axis=2)
    mn, mx = arr.min(), arr.max()
    rng = mx - mn if mx > mn else 1.0
    arr = (arr - mn) / rng
    flat = arr.flatten()
    if flat.shape[0] != 100:
        flat = np.resize(flat, 100)
    return flat

In [ ]:
Transition = namedtuple('Transition', ('state','action','reward','next_state','done'))

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    def push(self, *args):
        self.buffer.append(Transition(*args))
    def sample(self, batch):
        s = random.sample(self.buffer, batch)
        return Transition(*zip(*s))
    def __len__(self):
        return len(self.buffer)

class QNetworkMLP(nn.Module):
    def __init__(self, input_dim, hidden=[256,128], n_actions=6):
        super().__init__()
        layers = []
        last = input_dim
        for h in hidden:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU())
            last = h
        layers.append(nn.Linear(last, n_actions))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

In [ ]:
class ShapedBreakoutEnv:
    BALL_CH = 1  # correct ball channel

    def __init__(self):
        self.env = minatar.Environment("breakout")
        self.last_ball_y = None

    def reset(self):
        self.env.reset()
        st = self.env.state()
        self.last_ball_y = self._ball_y(st)
        return st

    def act(self, action):
        reward, done = self.env.act(action)
        st = self.env.state()

        ball_y = self._ball_y(st)
        shaping = 0.0

        # Shaping based on vertical motion of the ball
        if ball_y is not None and self.last_ball_y is not None:
            if ball_y < self.last_ball_y:      # ball moved UP
                shaping += 0.2
            elif ball_y > self.last_ball_y:    # ball moved DOWN
                shaping -= 0.2

        self.last_ball_y = ball_y

        return reward + shaping, done

    def state(self):
        return self.env.state()

    def _ball_y(self, st):
        arr = np.array(st)
        ball = arr[:,:,self.BALL_CH]
        pos = np.where(ball > 0)
        if len(pos[0]) == 0:
            return None
        return int(np.mean(pos[0]))

In [ ]:
def train_shaped_dqn(env,
                     steps=50000,
                     buffer_size=50000,
                     batch_size=64,
                     gamma=0.99,
                     lr=1e-3,
                     eps_start=1.0, eps_final=0.05, eps_decay=40000):

    replay = ReplayBuffer(buffer_size)

    state = preprocess_state(env.reset())
    n_actions = 6
    q = QNetworkMLP(100, [256,128], n_actions).to(device)
    target = QNetworkMLP(100, [256,128], n_actions).to(device)
    target.load_state_dict(q.state_dict())

    optim_q = optim.Adam(q.parameters(), lr=lr)

    def eps(t):
        return eps_final + (eps_start - eps_final) * max(0, 1 - t/eps_decay)

    rewards_log = []
    ep_reward = 0
    best = -9999

    for t in range(1, steps+1):
        if random.random() < eps(t):
            a = random.randint(0,5)
        else:
            with torch.no_grad():
                x = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
                a = int(torch.argmax(q(x)).item())

        r, done = env.act(a)
        next_state = preprocess_state(env.state())

        replay.push(state, a, r, next_state, float(done))

        ep_reward += r
        state = next_state

        if t % 400 == 0:
            rewards_log.append(ep_reward)
            ep_reward = 0

        if len(replay) > batch_size:
            batch = replay.sample(batch_size)
            s = torch.tensor(np.stack(batch.state), dtype=torch.float32).to(device)
            a_b = torch.tensor(batch.action, dtype=torch.long).unsqueeze(1).to(device)
            r_b = torch.tensor(batch.reward, dtype=torch.float32).unsqueeze(1).to(device)
            ns = torch.tensor(np.stack(batch.next_state), dtype=torch.float32).to(device)
            d = torch.tensor(batch.done, dtype=torch.float32).unsqueeze(1).to(device)

            q_values = q(s).gather(1, a_b)
            with torch.no_grad():
                q_next = target(ns).max(1)[0].unsqueeze(1)
                target_val = r_b + gamma*(1-d)*q_next

            loss = nn.MSELoss()(q_values, target_val)
            optim_q.zero_grad()
            loss.backward()
            optim_q.step()

        if t % 2000 == 0:
            target.load_state_dict(q.state_dict())
            print(f"[{t}] reward mean last 5: {np.mean(rewards_log[-5:]) if len(rewards_log)>5 else 'N/A'}")

    return q, rewards_log

In [ ]:
shaped_env = ShapedBreakoutEnv()
shaped_model, shaped_rewards = train_shaped_dqn(shaped_env, steps=30000)

[2000] reward mean last 5: N/A
[4000] reward mean last 5: 0.0
[6000] reward mean last 5: 0.0
[8000] reward mean last 5: 0.0
[10000] reward mean last 5: 0.0
[12000] reward mean last 5: 0.0
[14000] reward mean last 5: 0.0
[16000] reward mean last 5: 0.0
[18000] reward mean last 5: 0.0
[20000] reward mean last 5: 0.0
[22000] reward mean last 5: 0.0
[24000] reward mean last 5: 0.0
[26000] reward mean last 5: 0.0
[28000] reward mean last 5: 0.0
[30000] reward mean last 5: 0.0


In [ ]:
path_shaped = os.path.join(DRIVE_PROJECT_DIR, "shaped_dqn", "best_model.pth")
os.makedirs(os.path.dirname(path_shaped), exist_ok=True)
torch.save(shaped_model.state_dict(), path_shaped)
print("Saved shaped model:", path_shaped)

Saved shaped model: /content/drive/MyDrive/rl-final-project/shaped_dqn/best_model.pth


In [ ]:
def state_to_numpy(st):
    return np.array(st, dtype=np.float32)

In [ ]:
def shaped_policy(state):
    with torch.no_grad():
        x = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        return int(torch.argmax(shaped_model(x)).item())

import minatar
gif_env = minatar.Environment("breakout")

def generate_shaped_gif(env, steps=500, scale=20):
    frames = []
    env.reset()
    s = preprocess_state(env.state())
    for _ in range(steps):
        a = shaped_policy(s)
        _, _ = env.act(a)
        raw = state_to_numpy(env.state())
        img = raw.sum(axis=2)
        mn, mx = img.min(), img.max()
        img = ((img-mn)/(mx-mn+1e-8)*255).astype(np.uint8)
        frame = Image.fromarray(img).resize((200,200), Image.NEAREST)
        frames.append(np.array(frame))
        s = preprocess_state(env.state())
    path = os.path.join(DRIVE_PROJECT_DIR, "visuals", "shaped_rollout")
    os.makedirs(path, exist_ok=True)
    gif = os.path.join(path, "shaped_dqn.gif")
    imageio.mimsave(gif, frames, fps=10)
    print("Saved GIF:", gif)
    return gif

gif = generate_shaped_gif(gif_env)
gif

Saved GIF: /content/drive/MyDrive/rl-final-project/visuals/shaped_rollout/shaped_dqn.gif


'/content/drive/MyDrive/rl-final-project/visuals/shaped_rollout/shaped_dqn.gif'

In [ ]:
import matplotlib.pyplot as plt

original_curve = np.load(os.path.join(DRIVE_PROJECT_DIR, "plots", "dqn_rewards.npy")) if os.path.exists(os.path.join(DRIVE_PROJECT_DIR, "plots", "dqn_rewards.npy")) else None

plt.figure(figsize=(7,5))
plt.plot(shaped_rewards, label="Shaped DQN")
if original_curve is not None:
    plt.plot(original_curve, label="Original DQN")

plt.title("Reward Shaping: Learning Curve Comparison")
plt.xlabel("Training chunks")
plt.ylabel("Reward")
plt.legend()
plt.grid(True)

outpath = os.path.join(DRIVE_PROJECT_DIR, "plots", "shaping_comparison.png")
plt.savefig(outpath, dpi=150)
plt.close()

print("Saved shaping comparison plot:", outpath)

Saved shaping comparison plot: /content/drive/MyDrive/rl-final-project/plots/shaping_comparison.png


In [ ]:
import numpy as np
import minatar

env = minatar.Environment("breakout")
env.reset()
state = env.state()

print("state.shape:", state.shape)
print("unique channels ...")
for i in range(state.shape[2]):
    print(i, np.unique(state[:,:,i]))

state.shape: (10, 10, 4)
unique channels ...
0 [False  True]
1 [False  True]
2 [False  True]
3 [False  True]


In [ ]:
for ch in range( state.shape[2] ):
    if np.sum(state[:,:,ch]) > 0:
        print("Channel", ch, "has nonzero elements:", np.sum(state[:,:,ch]))

Channel 0 has nonzero elements: 1
Channel 1 has nonzero elements: 1
Channel 2 has nonzero elements: 1
Channel 3 has nonzero elements: 30


In [ ]:
import minatar
import numpy as np

env = minatar.Environment("breakout")
env.reset()

def find_ball_channel(env, steps=50):
    channels_movement = {0:[], 1:[], 2:[]}
    prev_positions = {}

    # initialize
    st = env.state()
    for ch in [0,1,2]:
        positions = np.where(st[:,:,ch] > 0)
        if len(positions[0]) > 0:
            prev_positions[ch] = positions

    for _ in range(steps):
        env.act(0)  # do nothing action
        st = env.state()
        for ch in [0,1,2]:
            positions = np.where(st[:,:,ch] > 0)
            if len(positions[0]) > 0:
                y = int(np.mean(positions[0]))
                channels_movement[ch].append(y)

    return channels_movement

mov = find_ball_channel(env)
print(mov)

{0: [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], 1: [4, 5, 6, 7, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], 2: [3, 4, 5, 6, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]}
